# Student Performance & Success Predictor — ML Experiments
**Hack-O-Week 5 & 6 — Machine Learning Project**

This notebook performs end-to-end Machine Learning on student academic performance:
1. **Exploratory Data Analysis (EDA)**
2. **Data Preprocessing & Leak-Free Pipeline**
3. **Task A: Regression** (Linear, Polynomial, Ridge, Lasso)
4. **Task B: Classification** (Logistic Regression, KNN)
5. **Model Comparisons & Interpretability**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Libraries loaded successfully.")

## 1. Data Exploration & EDA

In [ ]:
df = pd.read_csv("../data/student_data.csv")
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Data Types:")
print(df.info())
print("\nMissing Value Counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Target Distribution & Key Features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["Exam_Score"], kde=True, color="#2563EB", ax=axes[0])
axes[0].set_title("Distribution of Exam Score (Target)")
axes[0].set_xlabel("Exam Score")
axes[0].set_ylabel("Frequency")

sns.scatterplot(data=df, x="Hours_Studied", y="Exam_Score", alpha=0.5, color="#059669", ax=axes[1])
axes[1].set_title("Study Hours vs Exam Score")
axes[1].set_xlabel("Study Hours (per week)")
axes[1].set_ylabel("Exam Score")

sns.scatterplot(data=df, x="Attendance", y="Exam_Score", alpha=0.5, color="#DC2626", ax=axes[2])
axes[2].set_title("Attendance vs Exam Score")
axes[2].set_xlabel("Attendance (%)")
axes[2].set_ylabel("Exam Score")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap for Numerical Columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

## 2. Preprocessing & Train-Test Split (Zero Data Leakage)

In [ ]:
NUMERICAL_FEATURES = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions", "Physical_Activity"]
CATEGORICAL_FEATURES = ["Parental_Involvement", "Access_to_Resources", "Extracurricular_Activities", "Motivation_Level", "Internet_Access", "Family_Income", "Teacher_Quality", "School_Type", "Peer_Influence", "Learning_Disabilities", "Parental_Education_Level", "Distance_from_Home", "Gender"]

X = df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y_reg = df["Exam_Score"]
y_cls = (df["Exam_Score"] >= 65).astype(int) # 1: PASS, 0: FAIL

# 80/20 Train-Test Split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# ColumnTransformer Pipeline
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipe, NUMERICAL_FEATURES),
    ("cat", cat_pipe, CATEGORICAL_FEATURES)
])

# Fit strictly on train
X_train_r_trans = preprocessor.fit_transform(X_train_r)
X_test_r_trans = preprocessor.transform(X_test_r)

X_train_c_trans = preprocessor.fit_transform(X_train_c)
X_test_c_trans = preprocessor.transform(X_test_c)
print("Preprocessed training feature shape:", X_train_r_trans.shape)

## 3. Task A: Regression Models

In [ ]:
# 1. Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_r_trans, y_train_r)

# 2. Polynomial Regression (Degree 2)
poly_reg = Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("linear", LinearRegression())])
poly_reg.fit(X_train_r_trans, y_train_r)

# 3. Ridge Regression (L2 Regularization with CV)
ridge_reg = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge_reg.fit(X_train_r_trans, y_train_r)

# 4. Lasso Regression (L1 Regularization with CV)
lasso_reg = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0], cv=5, random_state=42)
lasso_reg.fit(X_train_r_trans, y_train_r)

print(f"Best Ridge Alpha: {ridge_reg.alpha_}")
print(f"Best Lasso Alpha: {lasso_reg.alpha_}")

In [ ]:
# Regression Evaluation
reg_models = {
    "Linear Regression": lin_reg,
    "Polynomial Regression": poly_reg,
    "Ridge Regression": ridge_reg,
    "Lasso Regression": lasso_reg
}

reg_results = []
for name, model in reg_models.items():
    preds = model.predict(X_test_r_trans)
    mae = mean_absolute_error(y_test_r, preds)
    mse = mean_squared_error(y_test_r, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_r, preds)
    reg_results.append({"Model": name, "MAE": round(mae, 4), "MSE": round(mse, 4), "RMSE": round(rmse, 4), "R2": round(r2, 4)})

reg_comp_df = pd.DataFrame(reg_results).sort_values(by="R2", ascending=False)
reg_comp_df

## 4. Task B: Classification Models

In [ ]:
# 1. Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_c_trans, y_train_c)

# 2. KNN with GridSearch tuning K in [3, 5, 7, 9, 11]
knn_grid = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [3, 5, 7, 9, 11]}, cv=5, scoring="f1")
knn_grid.fit(X_train_c_trans, y_train_c)
knn_best = knn_grid.best_estimator_
print("Best K for KNN:", knn_grid.best_params_["n_neighbors"])

In [ ]:
# Classification Evaluation
cls_models = {
    "Logistic Regression": log_reg,
    "K-Nearest Neighbors": knn_best
}

cls_results = []
for name, model in cls_models.items():
    preds = model.predict(X_test_c_trans)
    acc = accuracy_score(y_test_c, preds)
    prec = precision_score(y_test_c, preds)
    rec = recall_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds)
    cls_results.append({"Model": name, "Accuracy": round(acc, 4), "Precision": round(prec, 4), "Recall": round(rec, 4), "F1-Score": round(f1, 4)})

cls_comp_df = pd.DataFrame(cls_results).sort_values(by="F1-Score", ascending=False)
cls_comp_df

In [ ]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for idx, (name, model) in enumerate(cls_models.items()):
    preds = model.predict(X_test_c_trans)
    cm = confusion_matrix(y_test_c, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["FAIL", "PASS"], yticklabels=["FAIL", "PASS"], ax=axes[idx])
    axes[idx].set_title(f"{name} Confusion Matrix")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")
plt.tight_layout()
plt.show()